In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
############################################################
import os
from sklearn.model_selection import train_test_split

def load_genre_stems(root_path):
    data = {}
    genres = os.listdir(root_path)
    for genre in genres:
        genre_path = os.path.join(root_path, genre)
        if not os.path.isdir(genre_path):
            continue
        data[genre] = {}
        songs = os.listdir(genre_path)
        for song in songs:
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue
            stems = {}
            for file in os.listdir(song_path):
                if file.endswith(".wav"):
                    stem_name = file.replace(".wav", "")
                    stems[stem_name] = os.path.join(song_path, file)
            data[genre][song] = stems
    return data

# Step 1: Load real audio file paths
dataset_path = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
data = load_genre_stems(dataset_path)

# Step 2: Build recipe list from actual data (lazy - just paths, no audio loaded)
recipes = []
for genre, songs in data.items():
    for song_id, stems in songs.items():
        recipe = {
            "genre": genre,
            "song_id": song_id,
            "drums":  stems.get("drums"),
            "bass":   stems.get("bass"),
            "vocals": stems.get("vocals"),
            "other":  stems.get("other")
        }
        recipes.append(recipe)

print(f"Total recipes built from actual files: {len(recipes)}")

# Step 3: Split
train_recipes, val_recipes = train_test_split(
    recipes, test_size=0.2, shuffle=True, random_state=42
)

print(f"Train recipes: {len(train_recipes)}")
print(f"Val recipes:   {len(val_recipes)}")

# Peek at one recipe
print("\nSample recipe:")
print(train_recipes[0])

Total recipes built from actual files: 1000
Train recipes: 800
Val recipes:   200

Sample recipe:
{'genre': 'disco', 'song_id': 'disco.00089', 'drums': '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00089/drums.wav', 'bass': '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00089/bass.wav', 'vocals': '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00089/vocals.wav', 'other': '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00089/other.wav'}


In [4]:
################################################################
import numpy as np

SR = 16000
DURATION = 10
SAMPLES = SR * DURATION  # 16000 × 10 = 160,000

# Simulate 4 stems + 1 noise, all padded/truncated to exact length
drums  = np.random.randn(SAMPLES)
bass   = np.random.randn(SAMPLES)
vocals = np.random.randn(SAMPLES)
other  = np.random.randn(SAMPLES)
noise  = np.random.randn(SAMPLES)

# Verify each stem shape
for name, arr in zip(["drums", "bass", "vocals", "other", "noise"],
                     [drums, bass, vocals, other, noise]):
    print(f"{name:>6} shape: {arr.shape}")

# Sum 4 stems → mix
mix = drums + bass + vocals + other
print(f"\nAfter summing 4 stems → mix shape: {mix.shape}")

# Add scaled noise
noise_intensity = 0.2
final_mix = mix + noise_intensity * noise
print(f"After adding noise    → final_mix shape: {final_mix.shape}")

 drums shape: (160000,)
  bass shape: (160000,)
vocals shape: (160000,)
 other shape: (160000,)
 noise shape: (160000,)

After summing 4 stems → mix shape: (160000,)
After adding noise    → final_mix shape: (160000,)


In [5]:
#############################################################
import numpy as np
from transformers import AutoFeatureExtractor

# Step 1: Load the feature extractor
extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

# Step 2: Dummy audio — 160,000 ones
mix = np.ones(160000)

# Step 3: Pass through extractor
inputs = extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)

# Step 4: Extract and squeeze batch dimension
tensor = inputs["input_values"].squeeze(0)

print(f"input_values shape (with batch): {inputs['input_values'].shape}")
print(f"After .squeeze(0)              : {tensor.shape}")
print(f"\nAnswer: {list(tensor.shape)}")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

input_values shape (with batch): torch.Size([1, 1024, 128])
After .squeeze(0)              : torch.Size([1024, 128])

Answer: [1024, 128]


In [6]:
###########################################################
from transformers import ASTForAudioClassification

# Initialize AST with 10-class head
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Breakdown for clarity
total_params = sum(p.numel() for p in model.parameters())
frozen_params = total_params - trainable_params

print(f"Trainable parameters : {trainable_params:,}")
print(f"Frozen parameters    : {frozen_params:,}")
print(f"Total parameters     : {total_params:,}")
print(f"\nExact integer answer : {trainable_params}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable parameters : 86,196,490
Frozen parameters    : 0
Total parameters     : 86,196,490

Exact integer answer : 86196490


In [7]:
#####################################################################
import numpy as np

# Create test array
y_test = np.array([-0.85, 0.40, 0.20, -0.10])

# Apply normalization formula
y_normalized = y_test / (np.max(np.abs(y_test)) + 1e-9)

# Inspect step by step
print(f"y_test                  : {y_test}")
print(f"np.abs(y_test)          : {np.abs(y_test)}")
print(f"np.max(np.abs(y_test))  : {np.max(np.abs(y_test))}")
print(f"denominator (max + 1e-9): {np.max(np.abs(y_test)) + 1e-9}")
print(f"\ny_normalized            : {y_normalized}")
print(f"\nValue at index 0        : {y_normalized[0]}")
print(f"Rounded to 3 decimals   : {round(y_normalized[0], 3)}")

y_test                  : [-0.85  0.4   0.2  -0.1 ]
np.abs(y_test)          : [0.85 0.4  0.2  0.1 ]
np.max(np.abs(y_test))  : 0.85
denominator (max + 1e-9): 0.850000001

y_normalized            : [-1.          0.47058823  0.23529412 -0.11764706]

Value at index 0        : -0.9999999988235294
Rounded to 3 decimals   : -1.0
